# Sharper borders: cheap steps first, then the full retrain

Run in this order: (1) test-time augmentation, (2) CRF, (3) Gaussian tile stitching, (4) graph head, (5) retrain with a
full-resolution branch (V17).

**The same rule judges every step, fixed before the results** (`sharpness.passes_border_rule`): boundary F1 up by at least
0.02, pooled mIoU down by no more than 0.005, and no class IoU down by more than 0.05. Steps 1-3 and 5 are judged on
validation regions (2024). Only what passes goes on to the test regions, once.

**What to say in the report.** Sentinel-1 IW GRD pixels are 10 m, but the real resolution is about 20 m x 22 m, with speckle,
and the Dynamic World labels are noisy too. So claim about 20 m effective, not 10 m precision. A step that fails the rule is
still a result: it says the blur comes from the signal and the labels, not from the method.

In [ ]:
# 0. Paths. BASELINE is the model the steps are compared against.
import os, glob, shutil
BASE = globals().get('BASE', '/home/jovyan/vault/lulc'); OUT = globals().get('OUT', f'{BASE}/raw')
RUNS = globals().get('RUNS', f'{BASE}/runs')
os.chdir('/home/jovyan/vault/sar-lulc')
RUNS_MY = f'{BASE}/runs_multiyear'; os.makedirs(RUNS_MY, exist_ok=True)
BASELINE = 'V12_window3_balanced_stats'
print('cwd:', os.getcwd())
print('baseline seeds:', glob.glob(f'{RUNS_MY}/{BASELINE}/seed*/best.pt'))

In [ ]:
# 0b. Install the packages (needed again whenever the environment has been reset)
!pip -q install -r requirements.txt


## 1. Test-time augmentation (no retrain)
Averages the softmax over 8 flips and 90-degree rotations, the same 8 that training augments with. Inference takes 8 times as long.

In [ ]:
%run scripts/sharpen_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --year 2024 --settings tta --tag _tta

## 2. Convolutional CRF (no retrain)
Two settings, fixed in advance. Expect a small gain.

In [ ]:
%run scripts/crf_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --splits val --year 2024

## 3. Gaussian tile stitching (no retrain)
Weights overlapping tiles by distance from the tile edge instead of averaging them equally. Reduces seams. Expect a small effect.
If steps 1 and 3 both pass, the last line of the next cell scores them together.

In [ ]:
%run scripts/sharpen_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --year 2024 --settings gaussian --tag _gaussian
# %run scripts/sharpen_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --year 2024 --settings tta+gaussian --tag _both

## 4. Graph head (trains a small head on the frozen model)
Trained on validation regions, judged on 2025 test regions. **This step reads the test regions**, as the script is designed to,
so judge it once and do not tune it afterwards. Needs `scikit-image`. Expect a small gain.

In [ ]:
%run scripts/graph_head_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --train-splits val --eval-splits test test_ood --year 2024 --eval-year 2025

## 5. Full training: V17, a full-resolution branch (about a day, 3 seeds)
The U-Net's last decoder block upsamples from half resolution with no skip connection, so the raw radar never reaches the
output layer directly. V17 is V12 plus two 3x3 convolutions on the input, concatenated before the segmentation head. Nothing
else changes. The gain is unproven. `--workers 0` avoids the shared-memory crash. A finished seed is skipped, so if the
session stops, run the cell again.

In [ ]:
for v in ['V0_scratch_1m', 'V1_imagenet_1m']:      # the encoder-weights decision reads V0/V1 scores
    if os.path.isdir(f'{RUNS}/{v}') and not os.path.isdir(f'{RUNS_MY}/{v}'): shutil.copytree(f'{RUNS}/{v}', f'{RUNS_MY}/{v}')
%run scripts/train_all.py --data $OUT --out $RUNS_MY --variants V17_window3_fullres --seeds 0 1 2 --workers 0

Compare V17 with the baseline on the same validation regions and months, under the same rule.

In [ ]:
%run scripts/sharpen_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --year 2024 --settings none --compare-variant V17_window3_fullres --tag _v17

## 6. Confirm once on the test regions (2025 and 2026)
Run this only for what passed on validation, and only once. Edit the two variables first. For V17, `PASSED_STEP = 'none'` and
`PASSED_COMPARE = '--compare-variant V17_window3_fullres'`; otherwise leave `PASSED_COMPARE` empty. For a CRF that passed, use `crf_eval.py` with
`--splits test test_ood --year 2025`, then 2026.

In [ ]:
# Fill these in only for a step that PASSED on validation. Nothing has passed yet, so this cell stops by default.
PASSED_STEP = ''         # tta, gaussian or tta+gaussian; 'none' for V17
PASSED_COMPARE = ''      # '--compare-variant V17_window3_fullres' for V17
if not PASSED_STEP:
    raise SystemExit('Nothing passed on validation yet, so there is nothing to confirm on the test regions.')
for y in (2025, 2026):
    print(f'################ {y}')
    %run scripts/sharpen_eval.py --data $OUT --runs $RUNS_MY --variant $BASELINE --splits test test_ood --year $y --settings $PASSED_STEP $PASSED_COMPARE --skip-missing --tag _test_y{y}


## Decision
Adopt a step only if it passes on validation **and** still passes on both test years. If several pass, take the cheapest: no
retrain beats retrain. Picking the best of several rows on validation is mildly optimistic, which is why the test run is what counts.
If none passes, keep the current model and report that: the maps are limited by the radar's 20 m resolution and the labels.